# Guion de reunión y revisión reproducible

**FONDECYT 1240138 · guion fechado 16-sep-2026 · revisión del notebook 17-sep-2026.**

Este cuaderno conserva los once apartados del [guion crítico](guion_reunion_ACC_2026-09-16_revision_critica.md), corrige su trazabilidad y añade controles sobre archivos reales. Sigue el formato de `expand_pif`: cabecera Quarto, índice lateral, narrativa, tablas y celdas R rotuladas con tiempos. El discurso en presente pertenece al guion de la reunión prevista; no acredita que la reunión haya ocurrido ni que sus propuestas fueran aprobadas.

**Lectura de diez minutos:** sección 0, apertura (1), hoja de ruta (8) y acuerdos (10). Para revisión técnica, ejecutar las celdas en orden con un kernel R desde la raíz del proyecto o `__andres_control`.

**Qué se ejecuta:** lectura de registros, rejillas PIF, caché YPLL y tablas de auditoría; comprobaciones de esquema, soporte, política nula y concordancia de muertes; parseo sin ejecución del prototipo. No se recalculan RR, AAF, PIF, microdatos ni efectos de IB. Los controles del cuaderno pueden terminar correctamente y, a la vez, mostrar un diagnóstico científico **FAIL**. En particular, el caché YPLL requiere reconciliación antes de reportar nuevas cifras de años evitables.

**Convención de evidencia:** *verificado aquí* = lectura/control ejecutable de este cuaderno; *histórico* = registro de una corrida anterior; *propuesto* = decisión por acordar; *pendiente* = todavía no acreditado. Las fuentes L1–L9 siguen documentando la revisión original; no se afirma haber repetido todas sus lecturas primarias en esta sesión.

## 0. Cambios de esta revisión y autoridad vigente

La última entrada del [handoff canónico](codex_handoff_adam_rr_full_override_caveman.md) está fechada **24-jul-2026**. Una fecha de modificación posterior del archivo no convierte una nota antigua en hallazgo nuevo. Las notas del 14, 20 y 22 de julio deben leerse junto a esa actualización y al código activo.

| Tema | Conclusión para este cuaderno | Implicación |
| :--- | :--- | :--- |
| Motor y RR | `aaf_unified.R` calcula AAF/PIF; `rr_registry_adam.R` carga y organiza fuentes WHO/Adam. Table 5/PUC es una comparación separada para IHD/IS. | La memoria antigua que lo llamaba «candidato no cableado» quedó superada por las llamadas activas. No se reemplaza el RR principal por PUC. |
| Corridas y draws | El registro del 23–24 de julio documenta 65/65 celdas AAF y 29/29 PIF, validaciones y draws sincronizados main/PUC de ambas familias. | La ausencia histórica de draws AAF ya no describe estos archivos. Tener draws disponibles no verifica automáticamente toda tabla agregada. |
| Independencia AAF/PIF | PIF se calcula desde exposición, RR y sus propios sorteos; no consume la tabla AAF colapsada. | No atribuir a un intervalo AAF ausente un defecto de PIF. Tampoco asumir que semillas iguales prueban por sí solas sincronización. |
| Rejilla vigente | Los archivos `20260723` tienen 20.160 filas main, 8.400 aplicables y 1.792 PUC; las filas aplicables tienen resultados finitos y límites ordenados. | Las dos celdas NA señaladas para `20260715` no se observan en esta versión. Esto no valida los supuestos del modelo. |
| Soporte de escenarios | Volumen: 23 causas. HED/combinados: cinco causas aplicables. PUC: IHD/IS. | Comparar totales sobre soporte común; conservar explícitamente exclusiones y AAF=1. |
| YPLL | Se confirma diferencia de **cinco muertes en tres celdas** entre caché `20260714` y mortalidad/AAF `20260723`, sobre 1.188 llaves compartidas. | Es un fallo de concordancia, no una estimación nueva de la diferencia en YLL. No atribuir las cinco muertes a una sola corrección sin revisar registros. |
| IHD/PUC | La forma de RR a alta exposición explica parte de la divergencia con WHO/Adam. IHD femenina mantiene covarianza diagonal en el experimento. | La sensibilidad histórica con correlación negativa no recupera la covarianza real. `Fact` no autoriza reescalar gramos ni reparar intervalos. |
| Estado de consumo | El código activo incluye incertidumbre de RR de exbebedores en familias crónicas/CV y la desactiva en lesiones. Reducir volumen no elimina automáticamente todos los componentes FD/HED. | Volumen cero no equivale en general a AAF; curvas J permiten PIF negativos. No imponer dirección universal al efecto de lambda. |
| Edad | La exposición ENPG activa cubre 15–65; el grupo 4 se mapea al RR Adam 35–64. La base de muertes/YPLL conserva una banda 60+ abierta. | No confundir el límite de exposición con el dominio de mortalidad; la transferencia a mayores necesita decisión explícita. |

La rejilla incluye 16 escenarios y variantes de salida HED, incluidas seis parejas con `lambda=1` y configuración `rho=1`. Esos controles históricos no constituyen parámetros de una IB. El programa de IB requiere su propia cascada y comparador.

**Fuentes del estado activo:** [motor](aaf_unified.R), [registro RR](rr_registry_adam.R), [AAF](expand_pif.ipynb), [PIF](expand_pif2.ipynb), [experimento Table 5](aaf_table5_ihd_is_experiment.R) y las entradas fechadas del handoff. Se sintetizan sus implicaciones; el handoff permanece como registro canónico.

### Reglas locales: excepciones expresas

- **AGENTS.md, rutas:** faltan `.acc_root`, `_tools/paths.R` y `_tools/paths.py` en esta copia. Se usa `here::i_am()` sobre este notebook y luego `here::here()`, sin rutas absolutas persistidas. Es una adaptación limitada del mecanismo de raíz; no cambia datos ni configuración global.
- **MEMORY.md:** se descarta como estado vigente la nota «motor aún no cableado», contradicha por los notebooks activos y el handoff posterior. Se mantiene la consulta previa para cambios metodológicos; aquí no se implementa ninguno.
- **Permiso de notebooks:** la solicitud del usuario autoriza crear este archivo. No es necesario anular la restricción general de editar `.ipynb` sin permiso. `CLAUDE.md` remite a `AGENTS.md`; no hace falta anular ninguna otra regla. No se editan archivos de instrucciones ni memoria.

### Preparación e inventario

Se fijan versiones fechadas para que la evidencia no cambie silenciosamente al aparecer archivos nuevos. La existencia de los archivos de draws se verifica sin cargar sus matrices grandes; sus hashes completos no se revalidan en este cuaderno.

In [1]:
#| label: review-setup
#| results: "hold"
.t0 <- Sys.time()

review_packages <- c("here", "jsonlite", "knitr", "IRdisplay", "readxl", "dplyr", "purrr", "tibble")
review_missing <- review_packages[!base::vapply(review_packages, base::requireNamespace,
  base::logical(1), quietly = TRUE)]
if (base::length(review_missing)) base::stop("Missing packages: ", base::paste(review_missing, collapse = ", "))
base::suppressMessages(here::i_am("__andres_control/guion_reunion_ACC_2026-09-16_revision_critica.ipynb"))
review_stamp <- "20260723"
review_keys <- c("year", "gender", "age_group", "disease")
review_required <- c(
  "__andres_control/guion_reunion_ACC_2026-09-16_revision_critica.md",
  "__andres_control/codex_handoff_adam_rr_full_override_caveman.md",
  "__andres_control/pif2_pif_results_full_20260723.rds",
  "__andres_control/pif2_pif_results_table5_full_20260723.rds",
  "__andres_control/aaf_nested_by_disease_20260723.rds",
  "__andres_control/Mortality Estimates WHO 2024_20260723.xlsx",
  "Mortalidad/Matrices/YPLL_20260714.rds")
review_inventory <- base::data.frame(file = review_required,
  exists = base::file.exists(here::here(review_required)))
base::stopifnot(base::all(review_inventory$exists))
IRdisplay::display_html(base::as.character(knitr::kable(review_inventory, format = "html", row.names = FALSE)))

base::message(base::sprintf("review-setup elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))))


file,exists
guion_reunion_ACC_2026-09-16_revision_critica.md,TRUE
__andres_control/codex_handoff_adam_rr_full_override_caveman.md,TRUE
__andres_control/pif2_pif_results_full_20260723.rds,TRUE
__andres_control/pif2_pif_results_table5_full_20260723.rds,TRUE
__andres_control/aaf_nested_by_disease_20260723.rds,TRUE
__andres_control/Mortality Estimates WHO 2024_20260723.xlsx,TRUE
Mortalidad/Matrices/YPLL_20260714.rds,TRUE


review-setup elapsed minutes: 0.00



In [2]:
#| label: review-historical-runs
#| results: "hold"
.t0 <- Sys.time()

review_run_dir <- here::here("__andres_control", "manual_paf_pif_draws_20260723_001555")
review_runs <- base::lapply(c("expand_pif", "expand_pif2"), function(name) {
  result <- jsonlite::fromJSON(base::file.path(review_run_dir, base::paste0(name, "_result.json")))
  base::stopifnot(base::identical(result$status, "completed"),
    result$code_cells_completed == result$code_cells_total)
  base::data.frame(notebook = name, completed = result$code_cells_completed,
    total = result$code_cells_total, ended_at = result$ended_at)
})
IRdisplay::display_html(base::as.character(knitr::kable(base::do.call(base::rbind, review_runs), format = "html", row.names = FALSE)))
review_log <- base::readLines(here::here("__andres_control",
  "paf_draw_regeneration_manual_paf_pif_draws_20260723_001555_monitor.log"), warn = FALSE)
base::stopifnot(base::any(base::grepl("PASS validation expand_pif$", review_log)),
  base::any(base::grepl("PASS validation expand_pif2$", review_log)))
review_draw_files <- base::paste0(c(
  "aaf_synchronised_draws_who_adam_full_", "aaf_synchronised_draws_table5_puc_full_",
  "pif2_pif_synchronised_draws_full_", "pif2_pif_synchronised_draws_table5_full_"), review_stamp, ".rds")
review_draw_inventory <- base::data.frame(file = review_draw_files,
  data_present = base::file.exists(here::here("__andres_control", review_draw_files)),
  manifest_present = base::file.exists(here::here("__andres_control",
    base::sub("\\.rds$", ".manifest.rds", review_draw_files))))
base::stopifnot(base::all(review_draw_inventory$data_present), base::all(review_draw_inventory$manifest_present))
IRdisplay::display_html(base::as.character(knitr::kable(review_draw_inventory, format = "html", row.names = FALSE)))
base::cat("Historical logs: both validators recorded PASS. Draw inventory is an existence check only.\n")

base::message(base::sprintf("review-historical-runs elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))))


notebook,completed,total,ended_at
expand_pif,65,65,2026-07-23T00:58:29-04:00
expand_pif2,29,29,2026-07-23T14:08:36-04:00


file,data_present,manifest_present
aaf_synchronised_draws_who_adam_full_20260723.rds,TRUE,TRUE
aaf_synchronised_draws_table5_puc_full_20260723.rds,TRUE,TRUE
pif2_pif_synchronised_draws_full_20260723.rds,TRUE,TRUE
pif2_pif_synchronised_draws_table5_full_20260723.rds,TRUE,TRUE


Historical logs: both validators recorded PASS. Draw inventory is an existence check only.


review-historical-runs elapsed minutes: 0.00



## 1. Lo que conviene transmitir en la reunión

Tu posición más sólida es que has reducido incertidumbre sobre los datos y los métodos, y puedes convertir ese diagnóstico en un plan de implementación verificable. No necesitas presentar una hipótesis exploratoria como refutada ni afirmar que el modelo completo está terminado.

**Apertura sugerida, 60–90 segundos**

> «Mi trabajo reciente se ha concentrado en revisar la factibilidad y la consistencia de los insumos para el modelo. En la exploración cocaínas–alcohol encontré dos límites distintos: escasez de eventos al estratificar y falta de identificación de transiciones individuales con encuestas repetidas. Eso permite cerrar una ruta que no sería defendible como seguimiento individual, y deja infraestructura de armonización y diagnóstico que podemos reutilizar.

> El proyecto dispone de corridas de mortalidad y PIF con controles documentados, además del trabajo de elasticidad y del Delphi. La tarea siguiente es integrar esos componentes en una evaluación de política. Propongo priorizar el fortalecimiento del tamizaje y la intervención breve en APS, definir exactamente a quién se ofrece y contra qué atención habitual se compara, y construir primero un prototipo con controles explícitos. Después avanzamos hacia la proyección dinámica de diez años. Me gustaría acordar hoy ese alcance, los datos de implementación y un primer hito de dos semanas.»

**Cómo hablar del tiempo y de las contribuciones.** Si surge la menor dedicación al proyecto, reconocerla brevemente: «La tesis y los manuscritos redujeron mi disponibilidad; ahora propongo una dedicación y entregables acordados». No convertir la apertura en una lista de excusas. Separar tu aporte del avance colectivo: la existencia de un archivo no demuestra quién hizo cada contribución. Presentar las auditorías como controles que facilitan la integración, con ejemplos concretos y sin descalificar el trabajo de JRT.

**Tres mensajes que debes poder repetir sin notas**

1. **El diagnóstico de inviabilidad es un resultado útil:** evita estimar transiciones que los datos no identifican.  
2. **Intervención breve ya cabe dentro de la propuesta FONDECYT:** cambia el mecanismo de política que se prioriza, conservando el vínculo consumo–riesgo–salud.  
3. **La próxima entrega debe responder una decisión:** cuánto alcance, entrega y persistencia del efecto se necesitan para obtener un beneficio plausible, y qué incertidumbre domina esa respuesta.

## 2. Correcciones indispensables al borrador

| Afirmación del borrador | Evaluación crítica | Formulación defendible |
| :---- | :---- | :---- |
| «Dejé cerrado y validado el módulo de mortalidad» | Hay corridas completas del 23-jul y validación de artefactos registrada el 24-jul. Eso no valida todos los supuestos epidemiológicos ni la integración dinámica. | «Tenemos resultados y controles históricos documentados; hay controles pendientes antes de integrarlos». |
| «PIF para las 23 causas» sin distinguir escenarios | Los escenarios de volumen cubren 23 causas. HED y combinados tienen un conjunto más restringido. | Explicitar universo de causas por escenario; comparar sobre soporte común o mostrar componentes por separado. |
| «AAF=1 pendiente: hoy subestima lo evitable» | La omisión está documentada. AAF=1 no implica que una IB evite el 100% de esas muertes ni que el efecto neto sea una cota inferior demostrada. | «La estimación es parcial para las causas incluidas; falta parametrizar la respuesta de las causas totalmente atribuibles». |
| «YPLL validado» | Hay reconstrucción y reconciliación histórica; el caché utilizado precede una corrección de unidades de edad. El cotejo del notebook confirma cinco muertes de diferencia en tres estratos; falta reconstruir y reconciliar el caché. | «Reconciliar nuevamente YLL contra la base de muertes vigente es un control inicial». |
| «Desistimiento, no sustitución» | Las personas no se siguen en el tiempo; tampoco se observa toda la secuencia de problemas de ambas sustancias. | «Las asociaciones exploratorias no permiten establecer sustitución ni desistimiento individual». |
| «El Delphi no quiere precios» | Ronda 1: 97,3% para tamizaje/IB, pero también 78,4% para impuestos volumétricos y 75,7% para precio mínimo. | «IB tiene respaldo muy alto; precios permanece como línea complementaria». |
| «−12,3% y rebote a siete años» como parámetro central | Son supuestos históricos de Sheffield, sin validación directa para el programa chileno. | Usarlos como escenario de comparación; considerar efecto internacional, evidencia chilena y escenario incremental nulo. |
| «La base armonizada sirve directamente para transiciones» | Aporta diseño, demografía y distribuciones. El notebook de pseudo-panel no entrega un estado completo de gramos/día, AUDIT y HED ni transiciones observadas. | Reutilizar infraestructura; construir y validar el archivo de exposición necesario. |
| «No se ha tocado simulación desde abril» | Las fechas no prueban inactividad. Hay prototipos; el script de calibración revisado tiene un error de sintaxis confirmado. | «Hay código inicial, pero no se ha acreditado aquí una simulación chilena integrada y validada». |
| «EPF no permite estimar participación» | El diagnóstico depende de la definición de mercado y de la especificación examinada. | «Algunas especificaciones presentan separación y problemas de identificación; hay que acordar el estimando utilizable». |
| «Dos políticas terminadas en ocho semanas» | Omite dependencias de datos, medición, transición, capacidad, riesgos y validación. | Ofrecer hitos condicionados para un primer módulo de IB; precio se incorpora con una interfaz común después. |

**Interpretación del Delphi.** El 97,3% es acuerdo del panel, no cobertura, efectividad ni probabilidad de éxito de la intervención. La síntesis revisada incluye 37 participantes y define consenso desde 75%. Distingue el respaldo a un impuesto volumétrico general (78,4%) del ítem específico de aumentar impuesto por gramo (70,3%). No intercambiar ambos resultados. El documento anuncia una segunda ronda, pero por sí solo no acredita su estado operativo actual. [Fuente local: síntesis de ronda 1, introducción y tablas por dominio; L1.]

## 3. Qué puedes acreditar como avance

Mantener el orden del proyecto ayuda a que ACC reconozca inmediatamente cómo encaja cada contribución.

| Módulo | Evidencia disponible al revisar | Valor para el proyecto | Qué falta acreditar |
| :---- | :---- | :---- | :---- |
| **Mortalidad** | Corridas AAF/PIF de julio y validadores de esquemas, draws, manifiestos y controles internos. YLL/YPLL implementados. | Base para convertir exposición en carga y para establecer controles de reproducción. | Reconciliación vigente de YLL; coherencia temporal de RR y causas; integración con supervivencia. [L3–L5] |
| **Contrafactuales de política** | PIF de volumen y escenarios HED/combinados con cobertura de causas desigual; causas AAF=1 excluidas del PIF actual. | Motor de contraste de exposición ya explorado. | Sustituir reducciones impuestas por una cascada de intervención, igualar universos comparados y especificar retardos. [L4] |
| **Elasticidad** | Auditoría documentada de reconstrucción, definiciones de mercado y especificaciones. | Insumo complementario para el adaptador de precios. | Acordar con JRT versión del manuscrito, población, margen intensivo/extensivo y parámetros transferibles. [L7] |
| **Simulación** | Prototipos locales, script de calibración y SIMAH release 0.1.1, citado por DOI y no vendorizado en este repositorio; su DESCRIPTION declaraba versión de paquete 0.0.0.9000. | Arquitectura de referencia y punto de partida. | Depurar código, definir transiciones y población, calibrar y validar fuera del ajuste. [L8] |
| **Integración** | Existen componentes separados; no se acreditó una ejecución integral de IB chilena. | Es el próximo producto concreto. | Contrato de datos, ejecución basal/contrafactual, capacidad, persistencia, salud e incertidumbre en un flujo reproducible. |

No presentar las elasticidades numéricas del borrador como comprobadas en esta revisión: no se localizó el manuscrito citado como «Manuscript_IJDP_FINAL». Conviene llevar la versión que JRT considere vigente y confirmar si sus estimaciones corresponden a compradores, canales de compra y horizonte requeridos. Una elasticidad de participación en compra durante la ventana de EPF no equivale automáticamente a iniciar o abandonar consumo de alcohol.

**Hallazgos técnicos separados de las decisiones científicas**

- **Confirmado:** el parseo de [Alcohol Transitions_CALIB.R](../jrt/simulacion/Alcohol%20Transitions_CALIB.R) falla ante texto en prosa incrustado. También hay dos definiciones de calibrate_global y un uso posterior de componentes de resultado incompatible con la segunda definición. No se corrigió el archivo.  
- **Confirmado:** el chunk **pif2-ypll-three-metrics** carga YPLL_20260714.rds; los scripts activos incorporan después una corrección de unidades de edad. La reconciliación histórica registra 117.949 muertes, mientras el control vigente espera 117.944. **Verificado en este notebook:** sobre 1.188 llaves compartidas, el caché registra 117.949 muertes y los artefactos de mortalidad/AAF del 23-jul registran 117.944. Las cinco muertes de diferencia se concentran en tres celdas de 2024, grupo etario 1: LRI hombres (+2), LRI mujeres (+2) y lesiones no intencionales hombres (+1). Se cotejan conteos guardados; no se reconstruye YLL ni se determina aquí la causa a nivel de registro. [L5]  
- **Confirmado:** los escenarios PIF no comparten siempre el conjunto de causas. No comparar totales como si volumen, HED y combinación describieran exactamente el mismo desenlace. [L4]  
- **No acreditado:** terminación del motor dinámico nacional, cierre definitivo del Delphi, validación actual del manuscrito de elasticidad y ausencia de trabajo externo al repositorio. No inferir esas cosas a partir de fechas o de archivos ausentes.

### Control actual: soporte y política nula en PIF

Se inspeccionan las rejillas guardadas, sin correr el motor. Los controles exigen llaves únicas, siete olas, intervalos finitos y ordenados y baseline cero. Se permite PIF negativo: puede aparecer con funciones RR en J. Un intervalo finito no resuelve la covarianza ausente en IHD femenina PUC.

In [3]:
#| label: review-pif-support
#| results: "hold"
.t0 <- Sys.time()

review_main <- base::readRDS(here::here("__andres_control", "pif2_pif_results_full_20260723.rds"))
review_puc <- base::readRDS(here::here("__andres_control", "pif2_pif_results_table5_full_20260723.rds"))
review_pif_summary <- base::lapply(base::list(main = review_main, puc = review_puc), function(x) {
  key <- c("disease", "sex", "age_group", "year", "scenario_id")
  required <- c(key, "applicable", "pif", "pif_low", "pif_up", "n_sim")
  base::stopifnot(base::all(required %in% base::names(x)), !base::anyDuplicated(x[key]),
    !base::anyNA(x$applicable), base::setequal(x$year, base::seq(2012L, 2024L, 2L)))
  a <- x[x$applicable, ]
  base::stopifnot(base::all(base::is.finite(base::as.matrix(a[c("pif", "pif_low", "pif_up")]))),
    base::all(a$pif_low <= a$pif_up), base::all(a$pif <= 1),
    base::any(a$scenario_id == "baseline"),
    base::all(base::as.matrix(a[a$scenario_id == "baseline", c("pif", "pif_low", "pif_up")]) == 0))
  base::data.frame(rows = base::nrow(x), applicable = base::nrow(a),
    scenarios = base::length(base::unique(x$scenario_id)), n_sim = base::unique(x$n_sim))
})
IRdisplay::display_html(base::as.character(knitr::kable(base::do.call(base::rbind, review_pif_summary), format = "html")))
review_support <- stats::aggregate(disease ~ scenario_id, review_main[review_main$applicable, ],
  function(x) base::length(base::unique(x)))
base::names(review_support)[2] <- "applicable_causes"
base::stopifnot(base::all(review_support$applicable_causes[
  base::grepl("^volume_", review_support$scenario_id)] == 23L),
  base::all(review_support$applicable_causes[
    base::grepl("^(hed_|combined_)", review_support$scenario_id)] == 5L))
IRdisplay::display_html(base::as.character(knitr::kable(review_support, format = "html", row.names = FALSE)))
review_match_keys <- c("disease", "sex", "age_group", "year", "scenario_id")
review_main_cv <- review_main[review_main$disease %in% base::unique(review_puc$disease), ]
base::stopifnot(base::nrow(base::merge(review_main_cv, review_puc, by = review_match_keys)) == base::nrow(review_puc),
  base::nrow(review_main_cv) == base::nrow(review_puc))
base::cat("PUC female IHD non-baseline rows with upper PIF > 0.9: ",
  base::sum(review_puc$disease == "Ischaemic Heart Disease" & review_puc$sex == "female" &
    review_puc$scenario_id != "baseline" & review_puc$pif_up > 0.9), "\n", sep = "")

base::message(base::sprintf("review-pif-support elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))))


,rows,applicable,scenarios,n_sim
main,20160,8400,16,10000
puc,1792,1792,16,10000


scenario_id,applicable_causes
baseline,23
combined_v10_h25,5
combined_v10_h25_rt,5
combined_v20_h50,5
combined_v20_h50_rt,5
combined_v30_h50,5
combined_v30_h50_rt,5
hed_reduction_10,5
hed_reduction_10_rt,5
hed_reduction_25,5


PUC female IHD non-baseline rows with upper PIF > 0.9: 147


review-pif-support elapsed minutes: 0.00



### Control actual: concordancia de muertes del caché YPLL

Se reutiliza únicamente la definición de `ypll_pipeline_deaths()` de [ypll_icd_defs.R](ypll_icd_defs.R), extraída mediante el parser de R. No se ejecuta el resto del script ni sus configuraciones de rutas. Los dos argumentos de archivos se pasan explícitamente. El cociente `mort/AAF` conserva AAF negativas cuando corresponde; AAF cero no permite reconstruir el denominador y queda fuera de este soporte recuperable.

El criterio científico es **cero diferencias y cero llaves huérfanas dentro de las olas**. Un FAIL se muestra como resultado del diagnóstico, no se silencia ni se convierte en autorización para usar el caché. Las filas de años impares son esperadas porque YPLL cubre todos los años y PIF sólo las olas.

In [4]:
#| label: review-ypll-reconciliation
#| results: "hold"
.t0 <- Sys.time()

review_expr <- base::parse(file = here::here("__andres_control", "ypll_icd_defs.R"))
review_target <- base::vapply(review_expr, function(z) base::is.call(z) &&
  base::identical(z[[1]], base::as.name("<-")) &&
  base::identical(z[[2]], base::as.name("ypll_pipeline_deaths")), base::logical(1))
base::stopifnot(base::sum(review_target) == 1L)
review_fn_env <- base::new.env(parent = base::baseenv())
base::eval(review_expr[[base::which(review_target)]], envir = review_fn_env)
review_ypll <- base::readRDS(here::here("Mortalidad", "Matrices", "YPLL_20260714.rds"))
review_deaths <- review_fn_env$ypll_pipeline_deaths(
  xlsx = here::here("__andres_control", "Mortality Estimates WHO 2024_20260723.xlsx"),
  bundle = here::here("__andres_control", "aaf_nested_by_disease_20260723.rds"))
base::stopifnot(!base::anyDuplicated(review_ypll[review_keys]),
  !base::anyDuplicated(review_deaths[review_keys]),
  base::all(base::is.finite(review_deaths$n)),
  base::max(base::abs(review_deaths$n - base::round(review_deaths$n))) < 1e-9)
review_deaths$n <- base::round(review_deaths$n)
review_join <- base::merge(review_ypll, review_deaths, by = review_keys, all = TRUE)
review_both <- review_join[!base::is.na(review_join$deaths) & !base::is.na(review_join$n), ]
review_bad <- review_both[review_both$deaths != review_both$n, c(review_keys, "deaths", "n")]
review_orphans <- review_join[review_join$year %in% review_deaths$year &
  (base::is.na(review_join$deaths) | base::is.na(review_join$n)), ]
review_ypll_ready <- base::nrow(review_bad) == 0L && base::nrow(review_orphans) == 0L
review_reconciliation <- base::data.frame(shared_keys = base::nrow(review_both),
  cached_deaths = base::sum(review_both$deaths), current_deaths = base::sum(review_both$n),
  mismatched_keys = base::nrow(review_bad), wave_orphans = base::nrow(review_orphans),
  scientific_gate = if (review_ypll_ready) "PASS" else "FAIL")
IRdisplay::display_html(base::as.character(knitr::kable(review_reconciliation, format = "html", row.names = FALSE)))
IRdisplay::display_html(base::as.character(knitr::kable(review_bad, format = "html", row.names = FALSE)))
base::cat("YLL values were not rebuilt. A FAIL blocks reporting from this cached death base.\n")

base::message(base::sprintf("review-ypll-reconciliation elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))))


shared_keys,cached_deaths,current_deaths,mismatched_keys,wave_orphans,scientific_gate
1188,117949,117944,3,0,FAIL


year,gender,age_group,disease,deaths,n
2024,Hombre,1,Lower Respiratory Infection,26,24
2024,Hombre,1,Unintentional Injuries,422,421
2024,Mujer,1,Lower Respiratory Infection,23,21


YLL values were not rebuilt. A FAIL blocks reporting from this cached death base.


review-ypll-reconciliation elapsed minutes: 0.01



### Control actual: sintaxis del prototipo

El parseo comprueba sintaxis, no calibración ni consistencia de resultados. Si falla, se informa el error sin ejecutar el script. Las definiciones duplicadas de `calibrate_global` son un problema adicional de contrato que debe revisarse al reparar el prototipo.

In [5]:
#| label: review-prototype-parse
#| results: "hold"
.t0 <- Sys.time()

review_calib_path <- here::here("jrt", "simulacion", "Alcohol Transitions_CALIB.R")
review_parse <- base::tryCatch({
  base::parse(file = review_calib_path)
  base::data.frame(check = "Calibration script syntax", status = "PASS", detail = "Parse only; not executed")
}, error = function(e) {
  detail <- base::gsub(review_calib_path, "jrt/simulacion/Alcohol Transitions_CALIB.R",
    base::conditionMessage(e), fixed = TRUE)
  base::data.frame(check = "Calibration script syntax", status = "FAIL", detail = detail)
})
base::print(review_parse, row.names = FALSE)

base::message(base::sprintf("review-prototype-parse elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))))


                     check status
 Calibration script syntax   FAIL
                                                                                                                       detail
 Simulacion/Alcohol Transitions_CALIB.R:342:4: unexpected symbol\n341: comp_c <- get_comp(fit_c$calib)\n342: Un gr\n        ^


review-prototype-parse elapsed minutes: 0.00



## 4. Pseudo-paneles: explicar el aprendizaje sin exagerar el resultado

**Versión para decir en un minuto**

> «Exploré el desplazamiento cocaínas–alcohol con siete cortes transversales de ENPG. Al construir cohortes sintéticas, la baja frecuencia de problemas por cocaínas vuelve inestables las celdas. Además, aun con más muestra, las prevalencias en dos momentos no identifican quién pasó de un estado al otro. Por eso no corresponde presentar tasas individuales de transición. La armonización y el diagnóstico de precisión se pueden reutilizar, y las asociaciones sobre historia de consumo quedan como una línea descriptiva que requiere reproducción y un estimando más acotado.»

**Dos problemas diferentes.** La escasez de eventos produce imprecisión; aumentar muestra o agrupar celdas podría ayudar. La falta de identificación exige información longitudinal o restricciones adicionales. No desaparece simplemente usando un algoritmo más complejo.

**Qué números están respaldados por qué evidencia.** La tabla guardada de auditoría contiene 84 celdas cohorte de diez años × sexo × ola y mediana de ocho personas con uso de cocaínas en el último año. Ese desenlace no es consumo problemático. Los 558 casos de CP y mediana de cuatro del borrador constan en el handoff, pero no se localizaron las salidas que permitan reproducirlos de manera independiente. El grupo de 5.122 corresponde a duración desde inicio ≥5 años; las regresiones resumidas en el handoff usan 5.946 con duración positiva. No mezclar denominadores ni presentar sus coeficientes como resultados nuevamente validados. [L6]

**Por qué la edad de inicio ayuda, pero no resuelve todo.** Añade historia retrospectiva de exposición; no observa el comienzo del problema alcohólico, la trayectoria completa de cocaínas ni el momento de cese. La duración es edad actual menos edad de inicio. Rechazar que ambas edades actúen exclusivamente a través de esa diferencia cuestiona un modelo particular, no todas las formas posibles de sustitución. Una interacción transversal edad × exposición tampoco estima velocidad individual de desistimiento.

Las cotas de Fréchet ilustran el límite: con marginales p(C en t) y p(A en t+1), la probabilidad condicional de A entre quienes tenían C puede variar entre max(0, p(C)+p(A)−1)/p(C) y min(p(C),p(A))/p(C), si p(C)>0. Con C muy infrecuente, ese intervalo puede ser [0,1]. La afirmación «todos los pares de olas dan exactamente [0,1]» requiere la tabla por pares, no localizada aquí.

**Límites interpretativos adicionales:** selección por supervivencia y por residencia en hogares, recuerdo de edad de inicio, cambios de instrumento, composición de cohortes y confusión. La mortalidad atribuible a alcohol por edad/sexo no basta para corregir selección por historia de cocaínas. Se necesitarían datos o supuestos específicos. Un gradiente entre categorías nunca/ex/actual/problemático es compatible con varias explicaciones; no es por sí mismo una dosis-respuesta causal.

**Decisión editorial recomendada:** estacionar la línea fuera del camino crítico. Recuperarla sólo si hay una pregunta descriptiva independiente, análisis reproducible, medición defendible y tiempo asignado. No comprometer una revista ni anunciar «resultado negativo publicable» antes de ese control. El método de cohortes sintéticas de [Deaton (1985)](https://deaton.scholar.princeton.edu/publications/panel-data-time-series-cross-sections) no convierte cortes transversales en seguimiento de las mismas personas.

### Control actual: tabla de precisión del pseudo-panel

Este control reproduce la fila de cohortes de diez años × sexo × ola: **84 celdas y mediana de ocho usuarios de cocaínas en el último año**. No la renombra como consumo problemático ni como transiciones identificadas.

In [6]:
#| label: review-pseudopanel
#| results: "hold"
.t0 <- Sys.time()

review_pp <- utils::read.csv(here::here("__andres_control", "pseudopanel_fase1",
  "ppd_cell_audit_summary_20260805.csv"), check.names = FALSE)
review_pp_row <- review_pp[review_pp$design == "10y x sex x wave, all cohorts",
  c("design", "n_cells", "cocaines_median", "cocaines_pct_lt30")]
base::stopifnot(base::nrow(review_pp_row) == 1L,
  review_pp_row$n_cells == 84L, review_pp_row$cocaines_median == 8)
IRdisplay::display_html(base::as.character(knitr::kable(review_pp_row, format = "html", digits = 2, row.names = FALSE)))

base::message(base::sprintf("review-pseudopanel elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))))


design,n_cells,cocaines_median,cocaines_pct_lt30
"10y x sex x wave, all cohorts",84,8,85.71


review-pseudopanel elapsed minutes: 0.00



## 5. Qué significa realmente el cambio hacia intervención breve

La propuesta original ya contempla intervenciones psicosociales breves, modelamiento del consumo, mortalidad por causa y años de vida perdidos durante diez años. La novedad está en **priorizar y especificar la política**, no en reemplazar todos los objetivos. La propuesta distingue selección de políticas, desarrollo del modelo y evaluación de escenarios; la evaluación económica completa aparece como extensión futura. [L2, páginas 1, 8–11.]

**Pregunta propuesta para acordar con ACC**

> En la población adulta elegible para APS, ¿qué cambio en consumo, mortalidad por causas incluidas y años de vida perdidos se proyectaría durante diez años al fortalecer el tamizaje y la entrega de intervención breve, comparado con mantener la implementación habitual, bajo distintos niveles de cobertura, capacidad y persistencia del efecto?

Precisar primero qué significa «terapia breve»: consejo/intervención breve para consumo riesgoso, psicoterapia de varias sesiones o tratamiento de dependencia tienen poblaciones, profesionales, costos y efectos diferentes. La propuesta inicial de esta guía es **tamizaje e IB en APS para consumo riesgoso, con una rama separada de evaluación/derivación cuando corresponda**. El umbral de elegibilidad se definirá con el instrumento y protocolo elegidos; no se presume equivalencia entre AUDIT completo, AUDIT-C y diagnóstico de dependencia.

| Referencia | Qué conviene tomar | Qué no se transfiere automáticamente |
| :---- | :---- | :---- |
| **Sheffield/SAPM para IB** | Cascada de contacto, pesquisa, positividad, entrega, cambio de consumo y daño; capacidad asistencial y duración. | Coberturas inglesas/italianas, umbrales de pesquisa, entrega universal a positivos, efecto y duración. |
| **SIMAH/Kilian 2025** | Individuos heterogéneos, actualización anual, calibración, población abierta y separación del mecanismo de precios. | Parámetros estadounidenses, categorías, distribución por bebida, mortalidad y código sin adaptación. |
| **Proyecto chileno** | Diseño ENPG, exposición/RR, controles de mortalidad/PIF, datos locales y selección de escenarios. | Dar por resueltas las transiciones individuales o por validado el ciclo completo. |

Kilian et al. simulan consumo estadounidense y aplican los escenarios de precios en 2019; su publicación no constituye una validación de un programa chileno de IB con beneficios sanitarios acumulados durante diez años. Registrar versión, funciones reutilizadas y hash/revisión de cualquier adaptación local. [Kilian et al., 2025](https://pubmed.ncbi.nlm.nih.gov/40885207/).

**Ruta pragmática.** Puede construirse primero un contraste de exposición y daño con microdatos ponderados y después una simulación anual dinámica. No hace falta resolver cocaínas–alcohol para comenzar. La aplicación SAPM del estudio ODHIN reconoce que no representa trayectorias individuales de consumo a lo largo de toda la vida: es una referencia útil para separar evaluación de la intervención y reconstrucción de historias personales. El prototipo corto sería un producto intermedio; no cumpliría por sí solo el objetivo de microsimulación dinámica del FONDECYT. [Angus et al., ODHIN](https://academic.oup.com/eurpub/article/29/2/219/5098723).

## 6. Evidencia para parametrizar, con unidades y comparadores

| Fuente | Hallazgo que sirve al modelo | Traducción permitida y límite |
| :---- | :---- | :---- |
| **Kaner et al., Cochrane 2018** | Diferencia media de −20 g/semana a doce meses, IC95% −28 a −12, frente a intervención mínima o ninguna; heterogeneidad importante. | Escenario internacional de efecto absoluto. Equivale aritméticamente a aproximadamente −2,86 g/día; no es −20% ni una reducción nueva que se acumula cada año. [Fuente](https://www.cochrane.org/evidence/CD004148_effectiveness-brief-alcohol-interventions-primary-care-populations). |
| **Barticevic et al., Chile 2021** | Adultos, AUDIT 8–15; IB de cinco minutos por técnicos + folleto versus folleto. A seis meses, aproximadamente 80% versus 71% en bajo riesgo; p=0,1 para el desenlace primario; análisis de casos completos: 294 de 342 participantes aleatorizados. | Evidencia local de efecto incremental incierto. No demuestra equivalencia ni ausencia universal de beneficio. AUDIT no se transforma directamente en gramos. [Fuente](https://link.springer.com/article/10.1186/s13722-021-00248-4). |
| **Poblete et al., Chile 2017** | ASSIST y riesgo moderado; APS, urgencias y comisarías. IB versus folleto informativo: no se encontraron diferencias entre grupos en resultados de alcohol/drogas a tres meses; seguimiento completado por 62%. | Contexto local sobre efectividad e implementación, con población y medición distintas. ACC es coautor: su experiencia puede informar factibilidad. No es un parámetro AUDIT ni gramos/día. [Fuente](https://pubmed.ncbi.nlm.nih.gov/28239995/). |
| **Angus et al., Italia 2014** | SAPM aplica reducción de 12,3% y retorno lineal a basal a siete años; pesquisa AUDIT-C ≥4 mujeres/≥5 hombres. | Escenario histórico de modelación. Los puntos de corte no son automáticamente un protocolo chileno ni el efecto una estimación nacional. [Fuente](https://link.springer.com/article/10.1186/1471-2296-15-26). |
| **Purshouse et al., 2013** | En Inglaterra, el modelo proyecta aproximadamente 40% de cobertura a diez años al registro y 96% a siguiente consulta, con distinta carga de implementación. | Referencia de diseño y capacidad; no usar esos porcentajes como tasas anuales ni cobertura chilena. [Fuente](https://pubmed.ncbi.nlm.nih.gov/23015608/). |

**Duración:** el ensayo de Fleming citado para sostener persistencia observó 48 meses, con dos visitas y dos llamadas. No observó siete años de efecto. El retorno lineal a siete años es una extrapolación del modelo; tratar duración corta, intermedia y Sheffield histórico como supuestos estructurales separados. [Fleming et al., 2002](https://onlinelibrary.wiley.com/doi/10.1111/j.1530-0277.2002.tb02429.x).

**HED:** existe evidencia de cambios pequeños en frecuencia de episodios intensivos, pero frecuencia, prevalencia de HED y cantidad por ocasión no son el mismo desenlace. El metaanálisis no autoriza reducirlos todos en el mismo porcentaje. Armonizar definición y ventana antes de trasladar un efecto a los RR de lesiones. [Kaner et al., resultados](https://doi.org/10.1002/14651858.CD004148.pub4).

**Reglas para no construir un efecto ficticio**

- Elegir una escala principal: diferencia en gramos, cambio proporcional o cambio en un desenlace de riesgo con un puente validado. No sumar −20 g/semana, −12,3% y una OR chilena como si fueran beneficios independientes.  
- Si se aplica un efecto medio entre grupos, no multiplicarlo de nuevo por una probabilidad arbitraria de «responder» o por adherencia ya incluida en ese efecto. Una mezcla respondedores/no respondedores debe recuperar el mismo promedio de referencia.  
- El efecto Cochrane está medido a doce meses. La función temporal debe reproducir ese valor en ese punto; el patrón durante el primer año y después del seguimiento son supuestos adicionales.  
- Comparar fortalecimiento del programa con atención habitual real. Una IB versus folleto no equivale a IB versus nada; el descenso antes–después en ambos brazos tampoco identifica el efecto del folleto.  
- Mantener un escenario de efecto incremental nulo y sensibilidades de transportabilidad. No truncar automáticamente al beneficio una distribución de incertidumbre que admita ausencia de efecto o daño.

En Barticevic hay discrepancia entre el IC de un desenlace secundario AUDIT del resumen y el del cuerpo del artículo. Esta guía evita usar ese IC como parámetro. La evidencia chilena informa la incertidumbre local; todavía se necesita un vínculo defendible entre sus desenlaces y la exposición utilizada por los RR.

### Alcance del cotejo bibliográfico añadido

Se consultaron las fuentes primarias enlazadas para Kaner, Barticevic, Poblete, Angus, Fleming y Kilian. Se mantienen sus diferencias de población, comparador y desenlace. En Angus, la IB modelada dura diez minutos, mientras que el efecto de referencia proviene de intervenciones de duración media 24,9 minutos: la transportabilidad también depende de la intensidad. [Angus et al.](https://link.springer.com/article/10.1186/1471-2296-15-26). El seguimiento de Fleming fue de 48 meses. [Fleming et al.](https://pubmed.ncbi.nlm.nih.gov/11821652/).

**Referencia de arquitectura:** Kilian C, Buckley C, Lemp JM, Kou X, Kerr WC, Mulia N, Purshouse RC, Rehm J, Probst C. *Targeting alcohol use in high-risk population groups: a US microsimulation study of beverage-specific pricing policies*. Lancet Public Health, 2025. DOI [10.1016/S2468-2667(25)00165-3](https://doi.org/10.1016/S2468-2667(25)00165-3). Simula consumo en adultos estadounidenses de 18–79 años durante 2000–2019 y aplica precios en 2019. [Artículo](https://pmc.ncbi.nlm.nih.gov/articles/PMC12478644/).

**Procedencia SIMAH:** SIMAH release 0.1.1, citado por DOI y no vendorizado en este repositorio; su DESCRIPTION declaraba paquete `0.0.0.9000`. El artículo remite al depósito [Zenodo 10.5281/zenodo.15641639](https://doi.org/10.5281/zenodo.15641639). URL del repositorio upstream y commit no verificados aquí: deben registrarse antes de adaptar su código. Este cuaderno describe la referencia y no implementa ni valida una adaptación SIMAH chilena.

## 7. Arquitectura técnica: de la política al resultado sanitario

**Cadena central**

> Población y exposición basal → contacto APS → tamizaje → resultado y elegibilidad → IB recibida o derivación → cambio temporal en gramos/día y patrón HED → cambio de riesgo por causa → muertes y años de vida perdidos.

La política de precios tendría otra entrada —precio, participación y elasticidades— y compartiría la capa de exposición y salud. La elasticidad no es un requisito del módulo de IB.

### 7.1 Población, calendario y estado individual

**Propuesta para el primer prototipo:** adultos de la población efectivamente cubierta por los datos disponibles, con una intervención definida para APS y resultados explícitamente restringidos a ese marco. No llamarlo «impacto nacional en 15+» mientras falten mayores, población rural u otros grupos excluidos del marco. El notebook de pseudo-panel utiliza 15–65 como restricción operativa; confirmar límites y cambios de marco por ola antes de fijar el dominio final. Adolescentes necesitan una decisión y evidencia de intervención propias.

Resolver desde el principio qué ocurre al envejecer más allá de 65. Un modelo de población abierta que informa sólo edades 18–65 puede hacer salir a quienes alcanzan el límite, pero sus beneficios se limitarán a esa ventana. Seguir a una cohorte inicial 18–65 durante diez años exige exposición, RR y mortalidad para edades posteriores. No borrar esos años y después interpretar el resultado como beneficio de toda la vida. Mantener 15+ nacional como objetivo ampliado requiere fuentes adicionales y validación de su integración.

Separar tres calendarios: **estimación histórica, validación temporal y proyección de política**. Una corrida 2024–2034 no puede, por sí sola, reproducir 2012–2024. Además, si la decisión es implementar desde 2026, un inicio en 2024 sería una evaluación retrospectiva hipotética: acordar el año de intervención. Diez años son diez intervalos; 2024 a 2034 inclusive contiene once etiquetas anuales, no once años de seguimiento. La propuesta original eligió 2008–2019 para evitar distorsiones pandémicas; actualizarla a 2012–2024 exige justificar comparabilidad, no sólo aprovechar más olas. [L2, pp. 7–9.]

**Estado mínimo propuesto:** identificador, peso, edad, sexo, educación si su medición permite usarla, estado de consumo actual/ex/nunca cuando sea identificable, gramos/día, HED con su definición, variable de elegibilidad observada o imputada, tiempo desde última IB, historia necesaria para persistencia, estado vital. Guardar también contactos, pesquisas, IB y derivaciones para auditar la implementación. No añadir estados clínicos que no estén parametrizados.

### 7.2 Qué aportan las encuestas y cómo especificar las transiciones

Una distribución de categorías condicionada por edad, sexo y ola no es automáticamente una matriz de transición. En notación sencilla, p(t+1)=p(t)P(t) puede ser compatible con muchas matrices P. Un ajuste perfecto de marginales no identifica persistencia, recaída ni quién recibió intervenciones repetidas.

**Propuesta:** usar ENPG para población inicial y objetivos de calibración; documentar los supuestos de acoplamiento entre años y contrastar alternativas de persistencia compatibles con los datos. Priorizar evidencia longitudinal externa pertinente cuando exista. El rank matching y los modelos ordinales pueden ser componentes útiles, siempre presentados como construcciones del modelo. Si los datos tienen intervalos bienales, una probabilidad a dos años no se divide simplemente por dos para obtener la anual: ajustar un proceso anual compatible o tasas con una conversión explícita.

La propuesta menciona Langevin/Dutta. Ese método también impone supuestos: equilibrio local y fuerzas comparables, entre otros. No resuelve sin condiciones la identificación de historias. Dejarlo como alternativa metodológica a evaluar, no como una garantía de que el problema está solucionado. [Dutta et al., 2021](https://pubmed.ncbi.nlm.nih.gov/34804581/).

Para el prototipo anual, recomiendo un ciclo discreto documentado por su facilidad de auditoría y coherencia con las fuentes. Tiempo continuo puede ser preferible si la pregunta exige secuencias clínicas intranuales; no hace falta implementar dos motores para declarar validación. Acordar el cambio respecto a MicSim, planteado en la propuesta. Un segundo programa con iguales supuestos no reemplaza comparación con datos independientes.

### 7.3 Cascada y capacidad: cada porcentaje necesita denominador

Como identidad de auditoría, el número esperado de receptores de IB puede expresarse como:

**N × P(contacto) × P(tamizaje | contacto) × P(elegible | tamizaje) × P(IB recibida | elegible)**.

Las probabilidades son condicionales; la expresión no supone independencia. Calcular por grupos relevantes y después sumar, porque acceso, consumo y riesgo no están distribuidos uniformemente. «Positivo» y «elegible para esta IB» pueden ser etapas distintas. Si se usan sensibilidad/especificidad, definir respecto a qué condición se evalúan; no aplicar además una positividad observada como si fuera otro filtro independiente.

| Insumo a solicitar | Denominador y desagregación requeridos | Solución si no llega a tiempo |
| :---- | :---- | :---- |
| Población adscrita/elegible APS y contacto anual | Personas, edad, sexo, territorio y condición de acceso | Escenarios explícitos de contacto; no extrapolar visitas a personas únicas. |
| Tamizaje por alcohol | Instrumento, período, personas únicas y repetición | Mantener cobertura como variable de escenario. |
| Positividad y elegibilidad | Categoría de riesgo, regla de corte, denominador tamizado | Construir distribución conjunta con instrumento compatible o un puente externo validado. |
| IB efectivamente entregada | Elegibles, formato, duración, profesional, fidelidad | Variar entrega/capacidad independientemente del efecto clínico. |
| Derivación y acceso efectivo | Derivados, atendidos y tratamiento recibido | Contabilizar derivaciones; no asignar beneficio terapéutico sin evidencia propia. |
| Recursos | Minutos por pesquisa y por IB, profesionales, capacitación y cupos | Imponer escenarios de capacidad máxima y mostrar demanda no atendida. |

El informe oficial DIR-APS 2023 mezcla alcohol, tabaco y otras drogas y una población más amplia que la de los ensayos de IB en adultos. Sus totales de producción no identifican la cobertura específica de alcohol ni la entrega entre positivos elegibles. No dividir IB totales por tamizajes totales para obtener ese parámetro. Es un insumo de implementación, no una evaluación causal. Tampoco se verificó que 2023 sea el último año disponible. [DIPRES, monitoreo 2023](https://www.dipres.gob.cl/597/articles-337851_doc_pdf.pdf).

**Capacidad:** sumar tiempo de todos los tamizajes, incluidos negativos, IB, capacitación y seguimiento. El número de consultas no equivale a personas alcanzadas. Un programa puede aumentar prestaciones sin ampliar mucho el número de receptores únicos. Esa diferencia es una salida del modelo, no un detalle administrativo.

### 7.4 Exposición, persistencia y repetición

El modelo necesita consumo continuo además de categorías: una persona puede reducir gramos/día sin cambiar de categoría. Si los RR dependen de gramos, ese cambio puede importar. Mantener la relación entre volumen y HED y evitar combinaciones lógicamente incompatibles con la definición usada.

Para una política de efecto absoluto, una representación de trabajo es **consumo con política = max(0, consumo de referencia + diferencia asociada a IB)**. Para un efecto proporcional, usar su propia formulación. La diferencia depende del tiempo desde intervención y de los supuestos de mantenimiento. Compararla con la trayectoria de referencia contemporánea, no siempre con un consumo congelado en el año inicial.

**Medición y escala:** los puntos de corte AUDIT operan sobre respuestas al instrumento; los gramos utilizados en carga pueden incluir correcciones por subreporte. No asignar elegibilidad AUDIT directamente a gramos corregidos por ventas ni volver a aplicar esa corrección a un efecto clínico sin justificarlo. Se requiere un vínculo entre medición de encuesta, selección clínica y exposición usada por RR.

**Repetición:** registrar cuándo se tamizó/intervino, definir intervalo de repetición y si una nueva IB reemplaza, prolonga o incrementa un efecto anterior. Como primera especificación, evitar efectos acumulativos automáticos. No aplicar 12,3% cada año a toda persona tratada alguna vez. La disminución del efecto conductual, la reincidencia y el retardo del riesgo son procesos diferentes.

### 7.5 De exposición a muertes: el puente que debe quedar explícito

**Prototipo agregado de daño.** Para un mismo estrato, causa y año, definir M0 como promedio ponderado del RR bajo referencia y M1 bajo política, integrando volumen, HED y estado de consumo cuando la función lo requiera. Entonces:

**PIF = (M0 − M1) / M0.**

**Muertes potencialmente evitadas = muertes totales basales de esa causa y estrato × PIF.**

Esta relación es un contraste modelado condicionado a los supuestos de RR, comparabilidad y calendario del riesgo. No multiplicar adicionalmente por AAF: eso descontaría dos veces la carga. Tampoco usar AAF como probabilidad individual de morir. Verificar el control de política nula, que debe recuperar PIF=0, antes de producir escenarios.

**Microsimulación con mortalidad individual.** Si se decide modelar supervivencia y riesgos competitivos, derivar tasas individuales cuya agregación reproduzca las tasas observadas. Una parametrización candidata es tasa individual proporcional al RR, normalizada por el RR medio basal; necesita calibración y conversión correcta de tasas a probabilidades. Incluir otras causas de muerte y garantizar que una persona muera una sola vez. No combinar una capa de mortalidad completa con otra que vuelva a descontar las mismas muertes por PIF. Esta extensión requiere más validación que el puente agregado de corto plazo.

**Retardos:** las lesiones y las causas crónicas no tienen necesariamente el mismo tiempo de respuesta al cambio de consumo. Separar efectos a corto plazo de beneficios crónicos sujetos a historia de exposición y demora. No activar todo el beneficio de cáncer o cirrosis al año siguiente por defecto. Los resultados deben indicar horizonte y causas incluidas. Las funciones RR ya usadas por el proyecto son el punto de partida autorizado; esta guía no introduce nuevas causas ni modifica RR.

**AAF=1:** la atribución total de una causa no especifica cuánto cambia su riesgo con una IB. Hace falta una relación intervención–exposición–riesgo pertinente o mantenerla fuera del cálculo, identificando el alcance parcial. Nunca convertir automáticamente AAF=1 en PIF=1.

### 7.6 Años perdidos, equidad e incertidumbre

Elegir el estimando antes de comparar resultados. **YLL con tabla de vida** usa esperanza de vida restante a la edad de muerte; **YPLL con edad de referencia** usa max(edad de referencia−edad, 0). Son medidas distintas. Proponer YLL de tabla chilena como resultado principal y la métrica de la propuesta como comparabilidad, sujeto a acuerdo con ACC. Una reducción de YLL calculada sobre muertes en un horizonte limitado no equivale automáticamente a años de vida efectivamente ganados por las personas simuladas durante toda su vida. [L5.]

Reportar beneficios absolutos y por población, por sexo, edad y educación cuando los datos lo permitan. Incorporar diferencias de acceso y entrega antes de inventar heterogeneidad de eficacia clínica. Una política con beneficio total positivo puede ampliar brechas si llega menos a quienes tienen mayor riesgo.

Separar **variación Monte Carlo, incertidumbre de parámetros e incertidumbre estructural**. Los draws existentes de RR/PIF no cubren automáticamente contacto, elegibilidad, entrega, duración ni trayectoria. Usar escenarios emparejados y números aleatorios comunes donde sea viable para reducir ruido del contraste; verificar convergencia de la diferencia. Mantener dependencias entre parámetros y no llamar «IC observado» a cualquier intervalo de simulación. [ISPOR-SMDM, incertidumbre](https://www.ispor.org/publications/journals/value-in-health/abstract/Volume-15--Issue-6/Model-Parameter-Estimation-and-Uncertainty--A-Report-of-the-ISPOR-SMDM-Modeling-Good-Research-Practices-Task-Force-6).

## 8. Roadmap propuesto: hitos condicionados, con una salida si hay bloqueo

**Compromiso razonable para hoy:** un paquete de especificación y controles en dos semanas; un prototipo integrado de IB como objetivo de ocho semanas, condicionado a datos y validación; reservar una extensión a doce semanas si persisten cuellos de botella. Es una propuesta de trabajo desde el acuerdo de la reunión, no evidencia de que etapas previas estén completas ni garantía de resultados publicables.

| Etapa | Trabajo y producto verificable | Criterio para avanzar | Alternativa ante bloqueo |
| :---- | :---- | :---- | :---- |
| **Semanas 1–2: especificación y controles** | Ficha de política/comparador; población y calendario; registro de parámetros; inventario de archivos; reproducción de un resultado AAF/PIF existente y reconciliación de YLL; diagnóstico ejecutable del prototipo. | El control reproduce su referencia bajo la misma versión y configuración. El denominador de cada etapa de la cascada está definido. | Si el control no coincide, resolver la diferencia antes de confiar en nuevas cifras sanitarias. Se puede seguir trabajando en la especificación y en datos de implementación. |
| **Semanas 3–4: exposición y primer contraste** | Archivo analítico con consumo continuo/HED y elegibilidad compatible; modelo de implementación; escenario basal y alternativa a un año; diagrama y reporte de exclusiones. | Respeta pesos/marco, capacidad, balance de personas y exposición válida; política nula recupera referencia. | Si falta puente AUDIT–exposición, entregar demanda de tamizajes/IB y análisis de umbrales; no inventar muertes evitadas. |
| **Semanas 5–6: dinámica y calibración** | Ciclo anual con entradas/salidas, envejecimiento y persistencia; calibración histórica y validación temporal; alternativas de transición. | Ajuste por sexo/edad/consumo y mortalidad evaluado con métricas/tolerancias predefinidas; desempeño en datos no usados para ajuste. | Si falla, mantener el producto de corto plazo y diagnosticar medición/estructura; no extrapolar diez años como resultado final. |
| **Semanas 7–8: IB integrada** | Escenarios de cobertura/entrega, efecto y duración; exposición, causas incluidas, mortalidad/YLL y recursos; informe de incertidumbre. | Universo de causas comparable, retardo declarado, reconciliación sanitaria y error Monte Carlo aceptable respecto del contraste. | Entregar versión de investigación y señalar exactamente qué impide considerarla reportable. |
| **Semanas 9–12, si se requieren** | Resolver mayores de 65, causas totalmente atribuibles, disponibilidad de datos o latencias; revisión con colaboradores; protocolo y documentación para manuscrito. | Revisión externa de estructura, resultados y limitaciones; ejecución reproducible con datos reales. | Reprogramar alcance antes que ocultar exclusiones. |

La validación temporal debe reservar olas que no hayan guiado la selección del modelo. Si 2022/2024 ya se usaron para elegir especificaciones, describir ese uso y buscar una evaluación adicional o validación temporal repetida. Un buen ajuste global puede esconder errores en grupos prioritarios. No fijar una tolerancia numérica arbitraria después de ver los resultados. [ISPOR-SMDM, transparencia y validación](https://www.ispor.org/publications/journals/value-in-health/abstract/Volume-15--Issue-6/Model-Transparency-and-Validation--A-Report-of-the-ISPOR-SMDM-Modeling-Good-Research-Practices-Task-Force-7).

**Primera matriz de escenarios, sin inventar parámetros nacionales**

- **Referencia:** implementación habitual, con su incertidumbre documentada.  
- **Expansión:** más personas tamizadas, manteniendo entrega y formato de IB.  
- **Mejor entrega:** mayor proporción de elegibles recibe IB, manteniendo alcance.  
- **Combinado:** expansión y mejor entrega, limitado por capacidad.  
- Dentro de cada escenario: efecto incremental nulo; efecto internacional transportado con supuestos explícitos; escenario histórico Sheffield. La evidencia chilena sirve como contraste de transportabilidad y, cuando exista un puente válido, como escenario local propio.

No llamar automáticamente «políticas distintas» a variantes de cobertura de un mismo programa para cumplir el compromiso de varias políticas de la propuesta. Acordar con ACC cómo encajan en los productos comprometidos.

**Precio permanece en paralelo, fuera del camino crítico de IB.** Con JRT, fijar interfaz de bebida/precio/consumo, estimando de elasticidad y sensibilidad de participación. Falta de significación de cruzadas no demuestra que sean cero: su exclusión debe ser una decisión explícita de simplificación. Para comparar precios e IB, armonizar población, referencia, horizonte, causas, implementación y perspectiva. No prometer costo-efectividad sin costos y desenlaces adecuados; sí registrar desde ahora recursos asistenciales.

## 9. Objeciones previsibles y respuestas preparadas

| Pregunta de ACC | Respuesta breve y defendible |
| :---- | :---- |
| «¿Qué avanzaste concretamente?» | «Diagnóstico de factibilidad, armonización y controles de insumos. Puedo mostrar qué afirmaciones respaldan los archivos y cuáles requieren una nueva ejecución. Propongo convertirlo ahora en una entrega integrada». |
| «¿Entonces descartaste la hipótesis cocaína–alcohol?» | «Descarté estimar transiciones individuales con ese diseño sin supuestos adicionales. Las asociaciones exploratorias no prueban ausencia de sustitución». |
| «¿La microsimulación no necesita precisamente esas transiciones?» | «Necesita una dinámica especificada y validada para la pregunta de alcohol. Las encuestas fijan distribuciones; la persistencia se informa con evidencia adicional y sensibilidad. No depende de recuperar la secuencia cocaína–alcohol». |
| «¿Por qué IB si el ensayo chileno fue nulo?» | «El beneficio incremental local es incierto, especialmente frente a un control activo. El modelo permite explorar si un escalamiento sería útil bajo efectos plausibles, y qué dato haría cambiar esa conclusión». |
| «¿Sheffield ya resolvió esto?» | «Aporta la cascada y una arquitectura de evaluación. Debemos adaptar población, implementación, comparador, efectos y riesgos. Sus coberturas y duración no son datos chilenos». |
| «¿Cuánto moriría menos?» | «Todavía no hay una cifra defendible para esta política. Depende de alcance, entrega, efecto, persistencia, HED y latencias. El primer control será reproducir el resultado basal antes de generar un impacto nuevo». |
| «¿Qué falta realmente para empezar?» | «Acordar formato de IB y comparador, definir población y obtener denominadores por etapa. Mientras llegan datos podemos terminar los controles y construir escenarios de implementación». |
| «¿Por qué ocho semanas y no dos?» | «En dos semanas propongo especificación y controles; ocho es la meta para un prototipo integrado. La fecha de resultados reportables depende de superar validación y conseguir datos». |
| «¿La auditoría contradice a JRT?» | «Identifica condiciones y especificaciones que debemos conciliar. Revisaría con él la versión exacta y el estimando que necesita cada política antes de interpretar diferencias como contradicción». |
| «¿Qué publicamos primero?» | «El modelo y su validación cuando estén acreditados; después los escenarios de política. El estudio exploratorio de cocaínas queda separado y sujeto a reproducibilidad y prioridad». |

## 10. Decisiones que conviene cerrar hoy

**Priorizar cuatro acuerdos**, dejando los detalles de implementación para una revisión técnica:

1. **Política y comparador:** IB breve en APS para consumo riesgoso, otro formato terapéutico o tratamiento; fortalecimiento versus atención habitual. Confirmar con el equipo Delphi qué componentes concretos respaldan los expertos.  
2. **Alcance del primer producto:** población cubierta, año de intervención, resultado a corto plazo y ruta a diez años; manejo de adolescentes, mayores de 65 y causas excluidas.  
3. **Acceso a datos y responsables:** quién obtiene datos alcohol-específicos de APS, quién acuerda parámetros con el Delphi, quién integra modelo y quién valida exposición/daño. Propuesta de reparto: tú coordinas integración y trazabilidad; JRT acuerda insumos del módulo de precios; equipo Delphi traduce prioridad en especificación; ACC resuelve alcance y acceso. Confirmar disponibilidad antes de comprometer a otras personas.  
4. **Primer hito y dedicación:** revisión en dos semanas de ficha de política, registro de evidencia y controles reproducidos. Acordar tiempo protegido y qué trabajo queda estacionado.

**Preguntas técnicas de alto valor para ACC y colaboradores**

- A ACC: qué aprendizaje del ensayo de Poblete y del trabajo con servicios conviene incorporar al comparador y a las tasas de implementación.  
- Al equipo Delphi: si el consenso distingue pesquisa, consejo breve, capacitación, seguimiento y derivación; y si hay preferencias sobre población y modalidad.  
- A JRT: cuál es la versión vigente del manuscrito y qué reproduce el código compartido; cómo se interpretan participación, compras y consumo.  
- A Robin Purshouse, mediante coordinación de ACC: revisión de estructura y de la traducción local de la cascada, persistencia y retardos. La propuesta lo identifica como colaborador; eso no acredita disponibilidad inmediata. [L2, p. 11.]

**Cierre sugerido**

> «Si acordamos IB en APS como primera prioridad, mi próxima entrega será una especificación completa, la lista de parámetros con su respaldo y el control basal reproducido. Con eso podremos decidir qué escenarios sanitarios son defendibles y qué dato conviene conseguir primero. El avance de elasticidad queda conectado al mismo modelo para la etapa comparativa.»

## 11. Trazabilidad de la revisión y límites de lo verificado

**Clave de lectura:** «confirmado» significa cotejado contra el artefacto indicado; «histórico» corresponde a una corrida o tabla guardada; «propuesto» es una recomendación de diseño; «pendiente» no debe convertirse en hecho consumado.

| ID | Fuente exacta y localización | Qué respalda |
| :---- | :---- | :---- |
| **L1** | [Síntesis resultados ronda 1.docx](../Delphi/S%C3%ADntesis%20resultados%20ronda%201.docx), introducción y tablas de política de precios mínimos, impuestos y respuesta de servicios de salud. | Panel de 37, regla de consenso y porcentajes citados. La síntesis no acredita finalización de ronda 2. |
| **L2** | [Proposal Fondecyt 1240138.pdf](../Proposal%20Fondecyt%201240138.pdf), pp. 1, 6–11; tabla 2 cotejada en imagen de p. 11. | Objetivos, diez años, IB contemplada, población prevista, fuentes, Langevin/MicSim, colaboradores, calendario relativo y evaluación económica futura. El año administrativo actual requiere fecha efectiva del proyecto. |
| **L3** | [Registro de validación](paf_draw_regeneration_manual_paf_pif_draws_20260723_001555_monitor.log); [resultado de corrida AAF](manual_paf_pif_draws_20260723_001555/expand_pif_result.json); [resultado de corrida PIF](manual_paf_pif_draws_20260723_001555/expand_pif2_result.json); [validador](validate_paf_draw_regeneration.R). | Corridas históricas completadas el 23-jul, validación el 24-jul y naturaleza de los controles. No se ejecutaron nuevamente aquí. |
| **L4** | [expand_pif2.ipynb](expand_pif2.ipynb), chunk **pif2-tbl-averted-deaths**, texto de alcance y exclusión de causas totalmente atribuibles. | Distinción entre escenarios de volumen y conjunto de causas HED/combinados; alcance parcial. |
| **L5** | [YPLL_20260714.rds](../Mortalidad/Matrices/YPLL_20260714.rds); notebook anterior, chunk **pif2-ypll-three-metrics**; [build_ypll.R](build_ypll.R); [test_ypll_death_base.R](test_ypll_death_base.R); [ypll_icd_defs.R](ypll_icd_defs.R); [handoff RR/AAF](codex_handoff_adam_rr_full_override_caveman.md). | Caché cargado, definiciones de YLL/YPLL, corrección de edad y discrepancia entre control histórico y actual. Este notebook añade cotejo de conteos sobre 1.188 llaves compartidas y confirma cinco muertes de discrepancia; no recalcula el contenido YLL del caché. |
| **L6** | [Tabla guardada de auditoría de celdas](pseudopanel_fase1/ppd_cell_audit_summary_20260805.csv); [handoff pseudo-panel](../notes/handoffs_historicos/pseudopanel_deaton_handoff.md), especialmente §17; [arquitectura](pseudopanel_arquitectura_estudio.md); [notebook](pseudopanel_deaton_fase1.ipynb), chunks **ppd-setup**, **ppd-recode-helpers**, **ppd-build-pool**, **ppd-export-audit-tables** y **ppd-assertion-gate**. | Diagnóstico de celdas, diferencia entre CP y uso último año, denominadores y estado del código. El notebook actual no conserva salidas ejecutadas. Los scripts scratchpad citados para regresiones no se localizaron. |
| **L7** | [elasticidad_epf_handoff.md](../notes/handoffs_historicos/elasticidad_epf_handoff.md), apartados A1–A2 y diagnóstico posterior de Deaton. | Correcciones a generalizaciones iniciales y dependencia de la definición de mercado. No sustituye validar el manuscrito actual de JRT. |
| **L8** | [MWE.R](../jrt/simulacion/MWE.R); [Alcohol Transitions FINAL.R](../jrt/simulacion/Alcohol%20Transitions%20FINAL.R); [Alcohol Transitions_CALIB.R](../jrt/simulacion/Alcohol%20Transitions_CALIB.R); run_microsim_alt.R y simulate_mortality.R de SIMAH (release 0.1.1, DOI 10.5281/zenodo.15641639). | Existencia de prototipos y fuente de referencia. Se confirmó fallo de parseo del script de calibración; no se ejecutó una simulación completa. La referencia SIMAH no prueba equivalencia con Chile. |
| **L9** | [presentacion_micsim.qmd](../presentacion_micsim.qmd); [plan de fase siguiente](plan_fase_siguiente_micsim_2026-07-04.md). | Organización y planificación histórica; las marcas de completitud se contrastaron con evidencia más cercana a ejecución. |

**Precisión de términos:** ENPG corresponde al Estudio Nacional de Drogas en Población General de SENDA; EPF, a la Encuesta de Presupuestos Familiares de INE. No intercambiarlas. PAF/AAF describe carga atribuible bajo un contrafactual de referencia; PIF describe un cambio de exposición especificado. AUDIT es un instrumento; HED describe un patrón; ninguno equivale por sí solo a un diagnóstico de dependencia.

**Alcance del control realizado:** la revisión original fue documental y de código, con registros históricos, tablas guardadas y control sintáctico. Este notebook añade ejecución de controles de artefactos fechados, comparación de muertes del caché YPLL y diagnóstico sintáctico reproducible. No es una corrida reportable de microdatos, validación externa de la microsimulación ni estimación de muertes evitadas. Las cifras locales conservadas se atribuyen a su fuente; los números exploratorios sin trazabilidad suficiente se retiraron del discurso de apertura. Los controles sustantivos a ejecutar están especificados en el roadmap.

**Limitación principal de la propuesta técnica:** la precisión de una simulación no reemplaza evidencia sobre acceso, efectividad local, dinámica individual y latencia. La entrega debe mostrar cómo cambia la conclusión al cambiar esos supuestos. Esa transparencia es parte del resultado científico que conviene ofrecer a ACC.

## 12. Resultado de los controles y próxima acción

| Control | Qué permite concluir | Qué no permite concluir |
| :--- | :--- | :--- |
| Registros de ejecución e inventario | Hubo corridas completas y validadores históricos; están los cuatro archivos de draws y sus manifiestos. | Que todas las salidas hayan sido recalculadas aquí o se hayan revalidado los hashes. |
| PIF guardado | Soporte explícito, llaves únicas, baseline cero y valores finitos con límites ordenados en archivos `20260723`. | Eficacia de IB, correcta covarianza PUC o validez externa. |
| Muertes de YPLL | **FAIL:** cinco muertes de diferencia, tres estratos; no hay huérfanos dentro de las olas. | Magnitud de la diferencia en YLL ni reparación del caché. |
| Prototipo | **FAIL** sintáctico reproducible en `Alcohol Transitions_CALIB.R`. | Ausencia de otros avances de simulación dentro o fuera del repositorio. |
| Pseudo-panel | Se reproduce el diagnóstico de 84 celdas / mediana ocho usuarios. | Transiciones individuales ni los coeficientes exploratorios sin salidas reproducibles. |

**Acción inmediata recomendada:** acordar política/comparador y responsabilidades; reconciliar YPLL y reparar/validar el prototipo en tareas explícitas; sólo después producir resultados sanitarios nuevos. No se cambian supuestos del motor, no se agregan causas y no se aplican automáticamente las propuestas de esta guía.

Los FAIL son hallazgos del objeto revisado. Una ejecución limpia del notebook significa que los diagnósticos se reprodujeron, no que el pipeline sanitario haya pasado su validación.

In [7]:
#| label: review-session-info
#| results: "hold"
.t0 <- Sys.time()

base::print(utils::sessionInfo())

base::message(base::sprintf("review-session-info elapsed minutes: %.2f",
  base::as.numeric(base::difftime(Sys.time(), .t0, units = "mins"))))


R version 4.4.1 (2024-06-14 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 26200)

Matrix products: default


locale:
[1] C
system code page: 65001

time zone: America/Santiago
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] crayon_1.5.3      vctrs_0.6.5       cli_3.6.4         knitr_1.50       
 [5] rlang_1.1.5       xfun_0.51         purrr_1.0.4       generics_0.1.4   
 [9] textshaping_1.0.0 jsonlite_2.0.0    glue_1.8.0        rprojroot_2.0.4  
[13] htmltools_0.5.8.1 IRdisplay_1.1     IRkernel_1.3.2    ragg_1.3.3       
[17] readxl_1.4.5      cellranger_1.1.0  tibble_3.3.0      evaluate_1.0.5   
[21] fastmap_1.2.0     base64enc_0.1-3   lifecycle_1.0.4   compiler_4.4.1   
[25] dplyr_1.1.4       pkgconfig_2.0.3   here_1.0.1        systemfonts_1.2.1
[29] pbdZMQ_0.3-14     digest_0.6.37     R6_2.6.1          tidyselect_1.2.1 
[33] repr_1

review-session-info elapsed minutes: 0.00

